# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos: 
- Mario Mozo Ruiz
- Julián Campos Navarro


Problema:
> 1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. Combinar cifras y operaciones

Descripción del problema:(copiar enunciado)

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las
tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de
grabación independientemente del número de tomas que se graben. No es posible grabar más
de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:

Número de actores: 10

Número de tomas: 30

Actores/Tomas:

https://bit.ly/36D8IuK

(*) La respuesta es obligatoria





                                        

In [1]:
import pandas as pd
import numpy as np

# Cargamos los datos del archivo CSV, ignorando la primera fila como se indica
file_path = 'Datos problema doblaje(30 tomas, 10 actores) - Hoja 1.csv'
data = pd.read_csv(file_path, skiprows=1)
#display(data)
# Eliminamos las columnas 'Unnamed: 11' y 'Total'
data_cleaned = data.drop(columns=['Unnamed: 11', 'Total'])

# Seleccionamos solo las filas que corresponden hasta la toma 30
data_filtered = data_cleaned.iloc[:30]

# Mostramos los datos filtrados
display(data_filtered)


,Toma,1,2,3,4,5,6,7,8,9,10
0,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1,2,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
2,3,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,4,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
4,5,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
5,6,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
6,7,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
7,8,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
8,9,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
9,10,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




Nº de Bell: número de formas de dividir un conjunto de n elementos en  subconjuntos no vacíos y disjuntos, 8,46 * 10^23 para nuestro conjunto de 30 elementos.

In [2]:
#La restricción impuesta por el enunciado es que no se pueden grabar más de 6 tomas por día. Si ignoramos la restricción
#nos encontramos con que el rodaje puede realizarse en cualquier número de días entre 1 y 30. Consideraremos cada una de
#esas posibilildades y, dentro de cada una, cómo se pueden distribuir las tomas entre el número de días fijados. El
#número de particiones de un conjunto de n elementos es el n-ésimo número de Bell, por lo que vamos a calcularlo para
#determinar el número de posibilidades.

import numpy as np
import math

def bell(n):
    bell = [[0 for _ in range(n + 1)] for _ in range(n + 1)] #Matriz para el Triángulo de Bell.
    bell[0][0] = 1
    for i in range(1, n + 1):
        bell[i][0] = bell[i-1][i-1] #El primer elemento de cada fila es el último de la fila anterior.
        for j in range(1, i + 1):
            bell[i][j] = bell[i-1][j-1] + bell[i][j-1] #Rellenamos la fila sumando el elemento anterior y el de arriba.

    return bell[n][0] #El primer elemento de la fila n es el n-ésimo número de Bell.

print(f"El número de formas de dividir 30 tomas sin restricciones es: {bell(30)}")

#Ahora calculamos el número de posibilidades teniendo en cuenta las restricciones.

def bell_res(n, k):
    # a[i] guardará el número de particiones para un conjunto de tamaño i
    # respetando que ningún subconjunto supere el tamaño k_max
    a = [0] * (n + 1)
    a[0] = 1  # Caso base: un conjunto vacío tiene 1 forma de particionarse
    for i in range(1, n + 1):
        # El primer elemento debe ir en un grupo de tamaño 'j'
        # 'j' solo puede oscilar entre 1 y el mínimo entre (i) y nuestro límite (k_max)
        for j in range(1, min(i, k) + 1):
            # Elegimos j-1 compañeros para el primer elemento de entre los i-1 restantes
            combinaciones = math.comb(i - 1, j - 1)
            # Multiplicamos por las particiones posibles del resto (i-j)
            a[i] += combinaciones * a[i - j]

    return a[n]

print(f"El número de formas de dividir 30 tomas con las restricciones es: {bell_res(30, 6)}")

El número de formas de dividir 30 tomas sin restricciones es: 846749014511809332450147
El número de formas de dividir 30 tomas con las restricciones es: 726391948970868949621309


Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta



Para representar el espacio de soluciones del problema hemos elegido un **vector de 30 posiciones**, en la cual cada índice (del 0 al 29) representa una de las 30 tomas, y el valor almacenado en esa posición será el día asignado para grabar esa toma.

Por ejemplo, una solución que empiece por `[1, 1, 2, 3, 2, ...]` nos dice que la toma 0 y la 1 se graban el día 1, la toma 2 y la 4 el día 2, la toma 3 el día 3...

Esta elección tiene varias ventajas por las que la hemos elegido:

- **Evaluación de la función objetivo y las restricciones:** con la solución expresada como un vector podemos comprobar las restricciones de forma rápida y directa. Es fácil iterar sobre la lista para agrupar las tomas por día y obtener del CSV el número de actores distintos en cada día (es decir, evaluar la función objetivo). Otras estructuras más complejas (listas de listas o diccionarios) complican la extracción de los datos y la complejidad conceptual del problema.

- **Generación de soluciones vecinas a una dada:** esta estructura nos facilita mucho a la hora de generar soluciones vecinas. Cambiar una toma de un día a otro es tan simple como reasignar un valor en la lista: `solucion[toma] = nuevo_dia`.

  En este caso, la primera idea fue usar una estructura de listas agrupadas por días, pero entonces la operación de mover una toma de un día a otro se complica: tendríamos que escribir más código para borrarla de una lista e insertarla en otra, con el riesgo de cometer errores.

---

Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta

La función objetivo, $f(x)$, nos dirá el coste de una solución, es decir, cuántos desplazamientos deberemos pagar a los actores (recordamos que se paga un desplazamiento por cada día en el que un actor debe acudir al estudio de rodaje).
Según nuestra estructura de datos, la función consistirá en agrupar qué tomas caen en cada día (leyendo nuestro vector), buscar la información de cada toma en el DataFrame del CSV para ver qué actores participan en ella y contar cuántos actores distintos van ese día. La suma del número de actores distintos de cada día nos dará el valor final de la función objetivo para esa solución concreta.

Para definir matemáticamente esta función, primero establecemos las siguientes variables:
- $n = 30$ es el número total de tomas.
- $a = 10$ es el número total de actores.
- $x$ es el vector que representa la solución a evaluar, donde el valor $x_i$ representa el día asignado a la toma $i$.
- $D$ es el conjunto de todos los días programados en el vector $x$, es decir, días en los que se grabarán tomas.
- $M$ es la matriz binaria de datos (obtenida del CSV), donde $M_{i,j} = 1$ si el actor $j$ participa en la toma $i$, y $M_{i,j} = 0$ en caso contrario.

Para calcular el coste, necesitamos saber si el actor $j$ tiene que desplazarse al estudio en el día $d$. Definimos una variable $y_{d,j}$ que toma el valor $1$ si el actor trabaja ese día y $0$ si no. Esto se consigue buscando, entre todas las tomas asignadas a ese día $d$, si el actor participa en alguna de ellas:
$$y_{d,j} = \max_{\{i \mid x_i = d\}} M_{i,j}$$

Y, como hemos explicado anteriormente, la función objetivo $f(x)$ es la suma de todos los desplazamientos de los actores a lo largo de los días programados. Como queremos el menor coste posible, el objetivo del problema se define como:
$$\min f(x) = \sum_{d \in D} \sum_{j=1}^{a} y_{d,j}$$

Sujeto a la restricción de que el número máximo de tomas por día es 6:
$$\sum_{i=1}^{n} I(x_i = d) \leq 6 \quad \forall d \in D$$
(donde $I$ es la función indicadora que vale 1 si la toma $i$ se graba el día $d$, y 0 en caso contrario).


Respecto al tipo de problema, vemos claramente que se trata de un problema de **minimización**. El propio enunciado nos lo indica al pedirnos *"planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el **menor** posible"*. Como todos los actores cobran la misma cantidad fija por cada día que van al estudio (independientemente del número de tomas en las que participen ese día), minimizar el gasto equivale a minimizar el total de desplazamientos totales necesarios para grabar las 30 tomas.

---

Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

In [3]:
#Habiendo visto el número de posibilidades de nuestro problema, nos damos cuenta de que resolverlo
#utilizando fuerza bruta sería inviable. Sin embargo, vamos a crear un algoritmo que intente calcular
#todas las soluciones posibles y sus respectivos costes.

from itertools import combinations

def fuerza_bruta(df):
    """
    Implementación del método de fuerza bruta para el problema de doblaje.
    """
    # 1. Preparación de datos: Cada toma es un conjunto de actores
    # Extraemos la matriz de 30 tomas x 10 actores
    actores_por_toma = []
    for i in range(len(df)):
        # Identificamos qué actores (1-10) están en cada fila (toma)
        toma = {j for j in range(1, 11) if df.iloc[i, j] == 1}
        actores_por_toma.append(toma)

    def coste_sesion(indices_tomas):
        """Función objetivo local: cuenta actores únicos en una sesión."""
        actores_unicos = set()
        for idx in indices_tomas:
            actores_unicos.update(actores_por_toma[idx])
        return len(actores_unicos)

    def buscar_recursivo(tomas_pendientes):
        """
        Exploración exhaustiva de todas las particiones posibles.
        """
        # Caso base: no quedan tomas por asignar
        if not tomas_pendientes:
            return 0, []

        tomas_lista = list(tomas_pendientes)
        primera = tomas_lista[0]
        resto = tomas_lista[1:]

        mejor_coste = float('inf')
        mejor_plan = []

        # Generamos todas las combinaciones posibles para la sesión actual
        # (incluyendo siempre la 'primera' para evitar duplicados de particiones)
        # El límite es 6 tomas por sesión (1 fija + hasta 5 del resto)
        for r in range(min(len(resto) + 1, 6)):
            for combo in combinations(resto, r):
                sesion_actual = (primera,) + combo
                coste_actual = coste_sesion(sesion_actual)

                # Llamada recursiva para las tomas que aún no han sido asignadas
                tomas_restantes = tuple(set(resto) - set(combo))
                coste_restante, plan_restante = buscar_recursivo(tomas_restantes)

                coste_total = coste_actual + coste_restante

                # Si encontramos un coste menor, actualizamos el óptimo global
                if coste_total < mejor_coste:
                    mejor_coste = coste_total
                    mejor_plan = [sesion_actual] + plan_restante

        return mejor_coste, mejor_plan

    # Iniciamos la búsqueda con las 30 tomas (o las que tenga el DF)
    todas_las_tomas = tuple(range(len(actores_por_toma)))
    return buscar_recursivo(todas_las_tomas)

Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

Para calcular la complejidad del algotimo vamos a ir analizando las operaciones que realiza en sus distintos bloques:
1. Preparación de datos: El bucle `for` inicial recorre las $n$ filas leyendo los datos de 10 actores por toma. Con un total de $k \cdot n$ operaciones, la complejidad es de $O(n)$.

2. Exploración recursiva: En cada llamada a la función `buscar_recursivo`, con $k$ tomas pendientes, se utiliza la función `for combo in combinations(resto, r)` para generar subconjuntos, añadiendo a la toma inicial hasta 5 tomas a mayores. Para cada una de estas combinaciones se hace una nueva llamada recursiva. De esta forma, las operaciones dependen de $n$ ramas restantes en la primera iteración, de $n-1$ ramas en la segunda, y así hasta llegar a la última toma. De esta forma, al multiplicar las llamadas recursivas $n \cdot (n-1) \cdot ... \cdot 2 \cdot 1$ llegamos a una complejidad de $O(n!)$

Por órdenes de infinitud llegamos a que la complejidad del algoritmo es de $O(n!)$, ya que la complejidad lineal del primer punto queda completamente desestimada al lado del coste de la función de recursividad.

Para nuestro caso, con un valor de $n = 30$ tomas que debemos asignar, esta complejidad es inasumible desde el punto de vista computacional.

---

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

Este algoritmo mejora al de fuerza bruta porque un problema que originalmente era computacionalmente intratable (complejidad de $O(n!)$) se transforma en uno que se puede resolver en un tiempo asumible. Al utilizar un algoritmo voraz perdemos la seguridad de haber encontrado el óptimo global (la mejor solución), pero aún así se obtiene una solución muy buena, sobre todo respecto al tiempo empleado.

In [4]:
def algoritmo_voraz(df):

    # Convertimos el DataFrame en una lista de las tomas, en la que cada
    # elemento será un set con los actores que participan en esa toma.
    tomas_actores = []
    for i in range(len(df)):
        actores = set([j for j in range(1, 11) if df.iloc[i, j] == 1])
        tomas_actores.append(actores)


    tomas_restantes = list(range(len(tomas_actores)))

    # Vector de 30 posiciones inicializado a 0
    solucion = [0] * len(tomas_actores)

    dia_actual = 1
    coste_total = 0


    while tomas_restantes: # Mientras queden tomas por planificar
        tomas_hoy = []
        actores_hoy = set()

        # Para inicia un día nuevo (semilla), elegimos la toma que requiera MÁS
        # actores de las que todavía no hemos planificado, ya que será la que
        # más actores podrá tener en común con otras tomas.
        toma_inicial = max(tomas_restantes, key=lambda x: len(tomas_actores[x]))

        tomas_hoy.append(toma_inicial)
        actores_hoy.update(tomas_actores[toma_inicial])

        # La toma elegida se elimina de las restantes
        tomas_restantes.remove(toma_inicial)

        # Añadimos tomas al día hasta tener 6, eligiendo las que menos aumenten
        # el coste (nº desplazamientos <==> nº actores nuevos para ese día)
        while len(tomas_hoy) < 6 and tomas_restantes:

            mejor_toma = -1
            coste = float('inf')

            # Evaluamos todas las tomas candidatas a ser añadidas al día
            for candidato in tomas_restantes:

                # Coste (actores nuevos que añade esta toma al día)
                actores_candidato = tomas_actores[candidato]
                nuevos_actores = len(actores_candidato - actores_hoy)

                # Selección (elegimos la toma con el incremento mínimo)
                if nuevos_actores < coste:
                    coste = nuevos_actores
                    mejor_toma = candidato

            # Incorporamos la mejor toma candidata a este día
            tomas_hoy.append(mejor_toma)
            actores_hoy.update(tomas_actores[mejor_toma])
            tomas_restantes.remove(mejor_toma)


        # Añadimos las tomas del día al vector solución
        for toma in tomas_hoy:
            solucion[toma] = dia_actual

        # Sumamos a la función objetivo los actores que han participado hoy
        coste_total += len(actores_hoy)

        # Avanzamos al siguiente día
        dia_actual += 1

    return solucion, coste_total


solucion_voraz, coste_voraz = algoritmo_voraz(data_filtered)

print(f"Solución obtenida (vector de días): \n{solucion_voraz}\n")
print(f"Coste total: {coste_voraz} desplazamientos")
print(f"Número de días programados: {max(solucion_voraz)} días")

Solución obtenida (vector de días): 
[1, 1, 2, 2, 4, 1, 1, 3, 1, 4, 2, 3, 1, 3, 4, 5, 2, 3, 2, 5, 4, 3, 2, 3, 5, 5, 5, 4, 5, 4]

Coste total: 31 desplazamientos
Número de días programados: 5 días


(*)Calcula la complejidad del algoritmo

Respuesta

La complejidad del algoritmo voraz se expresará en función del tamaño de las entradas:
- $n$: número total de tomas ($n = 30$).
- $m$: número de actores ($m = 10$). Esta variable será, en un caso real, bastante más pequeña que el número de tomas, por lo que la trataremos como una constante.

Vamos a ir contando las operaciones por cada fragmento del algoritmo:
1. Preparación del DataFrame (bucle for): se recorren las $n$ filas del DataFrame, y por cada fila se evalúan las $m$ columnas para crear el conjunto de actores. En total, $n \cdot m$ operaciones.

2. Inicializaciones (hasta el bucle while): complejidad constante, $k$.

3. Bucle de asignación voraz (bucle while): se recorre $n$ veces, asignando las tomas una por una, aunque en cada iteración debe evaluar un candidato menos ($n$ candidatos la primera vez, $n-1$ la segunda...). Pongamos que el máximo de operaciones a realizar para elegir una toma que añadir al día actual es $c$ (bien sea una toma inicial del día con `max(tomas_restantes,...)` o una toma elegida para añadir a un día ya iniciado con `for candidato in tomas_restantes:`). De esta forma, realizamos un total de:
$$Total\_voraz = c \cdot n + c \cdot (n-1) + ... + 2c + c$$
$$Total\_voraz = c \cdot (n + n-1 + ... + 2 + 1)$$
$$Total\_voraz = c \cdot \frac{n(n+1)}{2}$$

En resumen, el número total de operaciones será:
$$Total = (n \cdot m) + k + c \cdot \frac{n(n+1)}{2}$$

Y aplicando la notación de la $O$ grande de Landau (eliminando los términos de menor orden), la complejidad del algoritmo es de $O(n^2)$.

Evidentemente, el coste de este algoritmo mejora claramente al de fuerza bruta, ya que pasamos de un coste de $O(n!)$, que es computacionalmente intratable, a un coste de $O(n^2)$, que se puede resolver en milisegundos para nuestros datos, y que incluso para valores mucho más grandes de $n$ se puede abordar sin problema.


Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

In [5]:
#Tomamos la semilla
np.random.seed(21)

#Generamos la matriz
#Ponemos probabilidad 50% de que un actor esté en una toma
# Esta probabilidad parece algo mayor que la del DataFrame de origen,
# por lo que el coste de las soluciones se incrementará
data = np.random.choice([0, 1], size=(30, 10))

#Ponemos que cada toma tenga al menos 1 actor
for i in range(30):
    if np.sum(data[i]) == 0:
        data[i, np.random.randint(0, 10)] = 1

#Creamos el datafreame
columnas_actores = [f'Actor {i+1}' for i in range(10)]
juego_aleatorio = pd.DataFrame(data, columns=columnas_actores)
juego_aleatorio.insert(0, 'Toma', range(1, 31))
juego_aleatorio

,Toma,Actor 1,Actor 2,Actor 3,Actor 4,Actor 5,Actor 6,Actor 7,Actor 8,Actor 9,Actor 10
0,1,1,1,0,0,0,0,0,1,0,0
1,2,1,1,0,0,0,1,0,1,1,1
2,3,1,0,0,0,0,0,0,0,1,0
3,4,1,0,0,1,1,0,1,0,0,0
4,5,0,0,0,0,1,1,0,0,1,0
5,6,0,1,1,0,0,0,1,0,0,0
6,7,0,1,0,1,0,1,1,1,1,0
7,8,1,0,0,0,0,0,0,1,0,0
8,9,0,0,0,0,1,1,1,1,0,0
9,10,1,1,1,0,0,0,1,0,0,1


Aplica el algoritmo al juego de datos generado

Respuesta

In [6]:
solucion_voraz, coste_voraz = algoritmo_voraz(juego_aleatorio)

print(f"Solución obtenida (vector de días): \n{solucion_voraz}\n")
print(f"Coste total: {coste_voraz} desplazamientos")
print(f"Número de días programados: {max(solucion_voraz)} días")

Solución obtenida (vector de días): 
[1, 3, 1, 1, 1, 1, 2, 2, 2, 3, 5, 3, 2, 3, 1, 3, 3, 2, 2, 4, 4, 5, 4, 4, 5, 4, 5, 5, 4, 5]

Coste total: 43 desplazamientos
Número de días programados: 5 días


Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

1. Statologos. (18 de octubre de 2022). [*Los números de Bell y el Triángulo de Bell*]. Recuperado el 6 de marzo de 2026, de https://statologos.com/campanas-numeros-campana-triangulo/

2. Python Software Foundation. (s.f.). [*itertools — Funciones que crean iteradores para un bucle eficiente*]. Documentación de Python 3.8. Recuperado el 6 de marzo de 2026, de https://docs.python.org/es/3.8/library/itertools.html

3. NumPy Developers. (s.f.). [*Random sampling. NumPy*]. Recuperado el 6 de marzo de 2026, de https://numpy.org/doc/stable/reference/random/index.html

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta

Para avanza en el estudio del problema, el primer paso podría ser aplicar algoritmos de Recocido Simulado o Algoritmos Genéticos para intentar modificar levemente la solución y ver si bajamos algo el coste.

Teniendo en cuenta variaciones reales del problema, se podrían tener en cuenta que algunas escenas deben seguir un orden determinado (para meterse más en el papel del personaje, y que el doblador pueda transmitir mejor las emociones por ejemplo), disponibilidad de los actores (es posible que un día concreto el actor 4 no esté disponible, por lo que las tomas que le involucren no se pueden grabar ese día) o necesidad de otros elementos (ruidos creados externamente con aparatos quizás).

Desde el punto de vista del incremento de las tomas (la variable $n$ que determina el coste del algoritmo), aunque la complejidad $O(n^2)$ sigue siendo asumible sin problemas, es muy probable que la distancia respecto a la solución óptima aumente considerablemente. Para un gran estudio de doblaje de películas y series, con decenas o cientos de dobladores, esto se puede traducir en un sobrecoste de miles de euros perdidos. En esta situación, aplicar técnicas de *Machine Learning* puede ser decisivo a la hora de reducir los costes.